# Análisis Integral de Ventas - Maven Roasters

**Periodo de análisis:** enero-junio de 2023  
**Objetivo:** profundizar el análisis exploratorio del desempeño transaccional de Maven Roasters, ampliando el cuaderno existente con una estructura más rigurosa, interpretaciones ejecutivas y estandarización del código.

---


### 1. Introducción

Este cuaderno retoma el trabajo preliminar ya desarrollado para Maven Roasters y lo extiende con un análisis exploratorio mucho más profundo. El propósito es convertir el notebook en un documento analítico de lectura ejecutiva: cada bloque combina código, tablas, visualizaciones e interpretación en español formal.

El análisis se orienta a responder tres preguntas de negocio: cómo se distribuyen las ventas, qué variables describen mejor el comportamiento del portafolio y qué relaciones entre atributos de producto, tiempo y tienda generan diferencias observables en cantidad, precio e ingreso por transacción.


### 2. Dataset overview

Primero se inspecciona la estructura del archivo fuente, se valida la hoja utilizada y se generan variables analíticas derivadas que facilitan una exploración más rica sin alterar la integridad de los datos originales.


In [ ]:
import warnings
import json
import nbformat
import numpy as np
import pandas as pd
from textwrap import dedent
import plotly.express as px
from IPython.display import Markdown, display
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

PALETA = ['#0466c8', '#0353a4', '#023e7d', '#002855', '#001845', '#001233', '#33415c', '#5c677d', '#7d8597', '#979dac']
RUTA_ARCHIVO = '/home/ubuntu/Uploads/coffee_shop_sales.xlsx'
RUTA_NOTEBOOK = '/home/ubuntu/maven_roasters_plan_analisis.ipynb'

MESES = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',6:'Junio',7:'Julio',8:'Agosto',9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}
DIAS = {0:'Lunes',1:'Martes',2:'Miércoles',3:'Jueves',4:'Viernes',5:'Sábado',6:'Domingo'}


def estilizar_figura(fig, titulo, x=None, y=None, leyenda=None, alto=520):
    fig.update_layout(
        title=dict(text=titulo, x=0.5, xanchor='center', font=dict(family='Arial Black', size=20)),
        font=dict(family='Arial', size=12),
        colorway=PALETA,
        template='plotly_white',
        height=alto,
        legend_title_text=leyenda,
        margin=dict(l=40, r=40, t=90, b=40)
    )
    if x:
        fig.update_xaxes(title_text=x)
    if y:
        fig.update_yaxes(title_text=y)
    return fig


def resumen_textual_serie(serie):
    return {
        'n': int(serie.shape[0]),
        'media': float(serie.mean()),
        'mediana': float(serie.median()),
        'q1': float(serie.quantile(0.25)),
        'q3': float(serie.quantile(0.75)),
        'min': float(serie.min()),
        'max': float(serie.max()),
        'sesgo': float(serie.skew()) if serie.nunique() > 2 else np.nan
    }


def tabla_iqr(serie):
    q1 = serie.quantile(0.25)
    mediana = serie.quantile(0.50)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    regiones = pd.Series(index=serie.index, dtype='object')
    regiones[serie < lim_inf] = 'Por debajo del bigote inferior'
    regiones[(serie >= lim_inf) & (serie < q1)] = 'Entre bigote inferior y Q1'
    regiones[(serie >= q1) & (serie < mediana)] = 'Entre Q1 y mediana'
    regiones[(serie >= mediana) & (serie < q3)] = 'Entre mediana y Q3'
    regiones[(serie >= q3) & (serie <= lim_sup)] = 'Entre Q3 y bigote superior'
    regiones[serie > lim_sup] = 'Por encima del bigote superior'
    tabla = regiones.value_counts(dropna=False).rename_axis('Rango IQR').reset_index(name='Conteo')
    orden = ['Por debajo del bigote inferior','Entre bigote inferior y Q1','Entre Q1 y mediana','Entre mediana y Q3','Entre Q3 y bigote superior','Por encima del bigote superior']
    tabla['Rango IQR'] = pd.Categorical(tabla['Rango IQR'], categories=orden, ordered=True)
    tabla = tabla.sort_values('Rango IQR').reset_index(drop=True)
    tabla['Porcentaje'] = (tabla['Conteo'] / len(serie) * 100).round(2)
    return tabla, regiones, {'q1': q1, 'mediana': mediana, 'q3': q3, 'iqr': iqr, 'lim_inf': lim_inf, 'lim_sup': lim_sup}


def insight_numerico(nombre, serie, tabla_regiones):
    stats = resumen_textual_serie(serie)
    mayor_bloque = tabla_regiones.sort_values('Conteo', ascending=False).iloc[0]
    outliers_altos = float(tabla_regiones.loc[tabla_regiones['Rango IQR'] == 'Por encima del bigote superior', 'Porcentaje'].fillna(0).sum())
    outliers_bajos = float(tabla_regiones.loc[tabla_regiones['Rango IQR'] == 'Por debajo del bigote inferior', 'Porcentaje'].fillna(0).sum())
    sesgo = 'asimetría positiva' if stats['media'] > stats['mediana'] else 'asimetría negativa' if stats['media'] < stats['mediana'] else 'distribución aproximadamente simétrica'
    return dedent(f"""
    **Lectura ejecutiva.** La variable **{nombre}** presenta una media de **{stats['media']:.2f}** y una mediana de **{stats['mediana']:.2f}**, lo que sugiere **{sesgo}**. El tramo con mayor concentración es **{mayor_bloque['Rango IQR']}** con **{mayor_bloque['Porcentaje']:.2f}%** de las observaciones. Los valores extremos representan **{outliers_bajos:.2f}%** por debajo y **{outliers_altos:.2f}%** por encima de los bigotes, por lo que la variable puede contener eventos poco frecuentes pero potencialmente relevantes para la operación.
    """).strip()


def frecuencia_categorica(df, columna):
    tabla = df[columna].astype(str).value_counts(dropna=False).rename_axis(columna).reset_index(name='Conteo')
    tabla['Porcentaje'] = (tabla['Conteo'] / len(df) * 100).round(2)
    return tabla


def tabla_para_grafico_categorico(tabla, columna, max_categorias=12):
    if len(tabla) <= max_categorias:
        return tabla.copy()
    top = tabla.head(max_categorias).copy()
    otros = pd.DataFrame({
        columna: ['Otros'],
        'Conteo': [tabla.iloc[max_categorias:]['Conteo'].sum()],
        'Porcentaje': [tabla.iloc[max_categorias:]['Porcentaje'].sum()]
    })
    return pd.concat([top, otros], ignore_index=True)


def columna_apta_wordcloud(serie):
    n_unicos = serie.nunique(dropna=False)
    promedio_largo = serie.astype(str).str.len().mean()
    return n_unicos <= 120 and promedio_largo >= 3


def insight_categorico(columna, tabla):
    top = tabla.iloc[0]
    diversidad = tabla[columna].nunique()
    return dedent(f"""
    **Lectura ejecutiva.** La variable **{columna}** contiene **{diversidad}** categorías observadas. La categoría más frecuente es **{top[columna]}**, con **{top['Conteo']:,} registros** (**{top['Porcentaje']:.2f}%** del total). La distribución permite evaluar concentración operativa, amplitud del portafolio y posibles dependencias del negocio respecto de unas pocas categorías dominantes.
    """).strip()


def resumen_relacion(df, cat_col, num_col):
    agrupado = df.groupby(cat_col, dropna=False)[num_col].agg(['count','mean','median','min','max']).reset_index()
    agrupado = agrupado.sort_values('mean', ascending=False)
    agrupado.columns = [cat_col, 'conteo', 'media', 'mediana', 'mínimo', 'máximo']
    return agrupado


def insight_relacion(cat_col, num_col, tabla):
    alto = tabla.iloc[0]
    bajo = tabla.iloc[-1]
    amplitud = alto['media'] - bajo['media']
    return dedent(f"""
    **Lectura ejecutiva.** La relación entre **{cat_col}** y **{num_col}** muestra diferencias visibles entre grupos. La categoría con mayor promedio es **{alto[cat_col]}** (**{alto['media']:.2f}**), mientras que la menor corresponde a **{bajo[cat_col]}** (**{bajo['media']:.2f}**). La brecha promedio de **{amplitud:.2f}** sugiere una asociación útil para segmentación comercial, priorización de portafolio o ajuste operativo.
    """).strip()


xls = pd.ExcelFile(RUTA_ARCHIVO)
hojas = xls.sheet_names
raw_df = pd.read_excel(RUTA_ARCHIVO, sheet_name=hojas[0])
df = raw_df.copy()
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['transaction_time'] = pd.to_datetime(df['transaction_time'].astype(str), format='%H:%M:%S')
df['revenue'] = df['transaction_qty'] * df['unit_price']
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day'] = df['transaction_date'].dt.day
df['day_of_week'] = df['transaction_date'].dt.dayofweek
df['hour'] = df['transaction_time'].dt.hour
df['month_name'] = pd.Categorical(df['month'].map(MESES), categories=['Enero','Febrero','Marzo','Abril','Mayo','Junio'], ordered=True)
df['day_name'] = pd.Categorical(df['day_of_week'].map(DIAS), categories=['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo'], ordered=True)
df['weekend_flag'] = np.where(df['day_of_week'] >= 5, 'Fin de semana', 'Día laboral')
df['time_of_day'] = pd.Categorical(pd.cut(df['hour'], bins=[5,10,13,16,20], labels=['Mañana','Mediodía','Tarde','Noche']), categories=['Mañana','Mediodía','Tarde','Noche'], ordered=True)

display(Markdown('#### Confirmación de carga y preparación analítica'))
resumen_archivo = pd.DataFrame({
    'Elemento': ['Archivo fuente', 'Hojas detectadas', 'Hoja utilizada', 'Filas', 'Columnas originales', 'Columnas analíticas finales', 'Fecha mínima', 'Fecha máxima', 'Ingresos totales'],
    'Valor': [RUTA_ARCHIVO, len(hojas), hojas[0], f"{raw_df.shape[0]:,}", raw_df.shape[1], df.shape[1], str(df['transaction_date'].min().date()), str(df['transaction_date'].max().date()), f"USD {df['revenue'].sum():,.2f}"]
})
display(resumen_archivo)
display(Markdown('**Insight.** El archivo contiene una sola hoja transaccional y, tras la ingeniería mínima de variables, el conjunto queda listo para un EDA más explicativo sin perder trazabilidad respecto de las columnas originales.'))

display(Markdown('#### Vista preliminar de registros'))
display(df.head(10))


### 3. Variable typing and classification

En esta sección se clasifican las variables originales y las derivadas en roles analíticos. Se separan explícitamente los identificadores —incluidos los que son numéricos— para evitar interpretaciones engañosas en correlaciones o relaciones estadísticas.


In [ ]:
registros = []
columnas_originales = list(raw_df.columns)
for col in columnas_originales:
    dtype = str(raw_df[col].dtype)
    if col in ['transaction_id', 'store_id', 'product_id']:
        clasificacion = 'Identificador'
        rol = 'Llave operativa'
        incluir = 'Excluir de correlación y relaciones estadísticas'
        nota = 'Es numérica por codificación, no por significado analítico.'
    elif 'date' in col:
        clasificacion = 'Fecha'
        rol = 'Temporal'
        incluir = 'Usar para derivar mes, día y estacionalidad'
        nota = 'Se analiza mejor mediante variables temporales derivadas.'
    elif 'time' in col:
        clasificacion = 'Hora'
        rol = 'Temporal'
        incluir = 'Usar para derivar hora y franja horaria'
        nota = 'Se excluye de correlación directa por formato temporal.'
    elif pd.api.types.is_numeric_dtype(raw_df[col]):
        clasificacion = 'Numérica'
        rol = 'Métrica'
        incluir = 'Incluir'
        nota = 'Variable cuantitativa interpretable para distribución y relaciones.'
    else:
        clasificacion = 'No numérica'
        rol = 'Categórica'
        incluir = 'Incluir'
        nota = 'Apta para análisis de frecuencias y segmentación.'
    registros.append({'Variable': col, 'Origen': 'Original', 'dtype': dtype, 'Clasificación': clasificacion, 'Rol analítico': rol, 'Regla de uso': incluir, 'Nota': nota})

derivadas = {
    'revenue': ('Numérica', 'Métrica de negocio', 'Incluir', 'Ingreso por línea transaccional; variable clave para interpretación comercial.'),
    'year': ('Numérica', 'Temporal derivada', 'Excluir', 'Constante en este dataset; no aporta variabilidad analítica.'),
    'month': ('Numérica', 'Temporal derivada', 'Excluir', 'Representa orden temporal, pero se interpreta mejor con month_name.'),
    'day': ('Numérica', 'Temporal derivada', 'Excluir', 'Ordinal de calendario; útil descriptivamente, no para correlación principal.'),
    'day_of_week': ('Numérica', 'Temporal derivada', 'Excluir', 'Codificación ordinal del día; se utiliza day_name para relaciones.'),
    'hour': ('Numérica', 'Temporal derivada', 'Incluir', 'Aproxima el momento del día y sí tiene lectura operativa.'),
    'month_name': ('No numérica', 'Temporal categórica', 'Incluir', 'Facilita lectura ejecutiva del patrón mensual.'),
    'day_name': ('No numérica', 'Temporal categórica', 'Incluir', 'Facilita comparación semanal.'),
    'weekend_flag': ('No numérica', 'Temporal categórica', 'Incluir', 'Resume comportamiento entre días laborales y fin de semana.'),
    'time_of_day': ('No numérica', 'Temporal categórica', 'Incluir', 'Resume el patrón intradía en franjas operativas.')
}
for col, valores in derivadas.items():
    registros.append({'Variable': col, 'Origen': 'Derivada', 'dtype': str(df[col].dtype), 'Clasificación': valores[0], 'Rol analítico': valores[1], 'Regla de uso': valores[2], 'Nota': valores[3]})

clasificacion_df = pd.DataFrame(registros)
display(clasificacion_df)

elegibles_numericas = ['transaction_qty', 'unit_price', 'revenue', 'hour']
elegibles_categoricas = ['store_location', 'product_category', 'product_type', 'product_detail', 'month_name', 'day_name', 'weekend_flag', 'time_of_day']
excluidas_analisis = ['transaction_id', 'store_id', 'product_id', 'transaction_date', 'transaction_time', 'year', 'month', 'day', 'day_of_week']

display(Markdown(f"**Insight.** Se identifican **{len(elegibles_numericas)} variables numéricas elegibles** y **{len(elegibles_categoricas)} variables no numéricas elegibles** para el EDA profundo. Las variables excluidas del análisis relacional principal son: **{', '.join(excluidas_analisis)}**."))


### 4. Global numeric analysis (correlation matrix, pairplot for relevant numeric variables only)

El análisis global se limita a variables numéricas con significado analítico directo. Se excluyen identificadores y códigos temporales para evitar correlaciones espurias.


In [ ]:
numeric_df = df[elegibles_numericas].copy()
correlacion = numeric_df.corr(numeric_only=True)
display(Markdown('#### Matriz de correlación de variables numéricas elegibles'))
display(correlacion)
fig_corr = px.imshow(correlacion, text_auto='.2f', color_continuous_scale=PALETA, aspect='auto')
fig_corr = estilizar_figura(fig_corr, 'Matriz de correlación: variables numéricas elegibles', 'Variable', 'Variable', alto=560)
fig_corr.show()

max_corr = correlacion.where(~np.eye(correlacion.shape[0], dtype=bool)).stack().sort_values(key=lambda s: s.abs(), ascending=False)
par_top = max_corr.index[0]
valor_top = max_corr.iloc[0]
display(Markdown(f"**Insight.** La asociación lineal más marcada aparece entre **{par_top[0]}** y **{par_top[1]}** con una correlación de **{valor_top:.2f}**. Esto es coherente con la construcción de ingresos y con la relación económica entre precio unitario, cantidad e ingreso por transacción."))

display(Markdown('#### Matriz de dispersión (equivalente a pairplot)'))
muestra_pairplot = numeric_df.sample(min(5000, len(numeric_df)), random_state=42)
fig_pair = px.scatter_matrix(muestra_pairplot, dimensions=elegibles_numericas, opacity=0.35, color_discrete_sequence=PALETA)
fig_pair.update_traces(diagonal_visible=False, showupperhalf=False)
fig_pair = estilizar_figura(fig_pair, 'Matriz de dispersión de variables numéricas elegibles', alto=780)
fig_pair.show()

display(Markdown('**Interpretación.** La matriz de dispersión permite distinguir relaciones lineales, concentraciones por rangos discretos y presencia de valores extremos. En este caso, la nube asociada a `revenue` refleja combinaciones repetidas de cantidades discretas y precios de catálogo.'))


### 5. Univariate analysis for numeric variables

Para cada variable numérica elegible se presentan box plot, tabla de rangos IQR, histograma por regiones del IQR e interpretación de negocio. Este bloque profundiza la lectura de dispersión, concentración y eventos atípicos.


In [ ]:
for columna in elegibles_numericas:
    serie = df[columna].dropna()
    tabla_regiones, regiones, limites = tabla_iqr(serie)
    display(Markdown(f"#### Variable numérica: {columna}"))
    descripcion = pd.DataFrame([resumen_textual_serie(serie)])
    display(descripcion)

    fig_box = px.box(df, y=columna, color_discrete_sequence=[PALETA[0]], points='outliers')
    fig_box = estilizar_figura(fig_box, f'Box plot de {columna}', y=columna)
    fig_box.show()

    display(Markdown('##### Tabla de rangos asociados al IQR'))
    display(tabla_regiones)

    hist_df = df[[columna]].copy()
    hist_df['Rango IQR'] = regiones.astype(str)
    fig_hist = px.histogram(hist_df, x=columna, color='Rango IQR', nbins=40, color_discrete_sequence=PALETA)
    fig_hist = estilizar_figura(fig_hist, f'Histograma de {columna} segmentado por rangos IQR', x=columna, y='Frecuencia', leyenda='Rango IQR')
    fig_hist.show()

    display(Markdown(insight_numerico(columna, serie, tabla_regiones)))
    display(Markdown(f"**Contexto de negocio.** En `{columna}` conviene monitorear los extremos porque pueden representar tickets excepcionalmente altos, productos premium, compras multipaquete o concentraciones operativas en ciertas franjas horarias."))


### 6. Univariate analysis for non-numeric variables

Cada variable no numérica elegible se analiza mediante tabla de frecuencias, gráfico de barras y nube de palabras cuando el contenido es apto. Si la naturaleza de la variable no hace recomendable la nube de palabras, se deja constancia explícita.


In [ ]:
for columna in elegibles_categoricas:
    display(Markdown(f"#### Variable no numérica: {columna}"))
    tabla_freq = frecuencia_categorica(df, columna)
    display(tabla_freq)

    tabla_chart = tabla_para_grafico_categorico(tabla_freq, columna, max_categorias=12)
    fig_bar = px.bar(tabla_chart, x=columna, y='Conteo', color=columna, color_discrete_sequence=PALETA)
    fig_bar = estilizar_figura(fig_bar, f'Distribución de frecuencias de {columna}', x=columna, y='Conteo', leyenda=columna, alto=560)
    fig_bar.update_layout(showlegend=False)
    fig_bar.show()

    if columna_apta_wordcloud(df[columna].astype(str)) and columna not in ['store_location']:
        texto = ' '.join(df[columna].astype(str).tolist())
        nube = WordCloud(width=1200, height=600, background_color='white', colormap='Blues').generate(texto)
        imagen = np.array(nube)
        fig_wc = px.imshow(imagen)
        fig_wc = estilizar_figura(fig_wc, f'Nube de palabras de {columna}', alto=520)
        fig_wc.update_xaxes(showticklabels=False)
        fig_wc.update_yaxes(showticklabels=False)
        fig_wc.update_layout(coloraxis_showscale=False)
        fig_wc.show()
        display(Markdown('**Justificación.** La nube de palabras es pertinente porque la variable contiene descripciones semánticas legibles y permite visualizar concentración textual del portafolio.'))
    else:
        display(Markdown('**Nube de palabras omitida.** La variable tiene muy baja cardinalidad o representa etiquetas demasiado compactas para que la nube agregue valor interpretativo adicional.'))

    display(Markdown(insight_categorico(columna, tabla_freq)))
    display(Markdown(f"**Contexto de negocio.** La distribución de `{columna}` ayuda a identificar concentración de demanda, amplitud del surtido y posibles dependencias comerciales de una parte limitada del catálogo o de determinados momentos de operación."))


### 7. Interaction analysis: non-numeric vs numeric variables (split into Business Relevant Relations and All Other Relations)

Se construye un inventario completo de relaciones entre variables categóricas y métricas numéricas. A partir de ese inventario se separan las relaciones con mayor valor interpretativo de aquellas que, aunque menos prioritarias, siguen siendo útiles para una exploración exhaustiva.


In [ ]:
relaciones_prioritarias = [
    ('store_location', 'revenue'),
    ('product_category', 'revenue'),
    ('product_type', 'unit_price'),
    ('month_name', 'revenue'),
    ('day_name', 'revenue'),
    ('time_of_day', 'revenue'),
    ('weekend_flag', 'revenue'),
    ('product_category', 'transaction_qty')
]

universo_relaciones = [(c, n) for c in elegibles_categoricas for n in elegibles_numericas]
relaciones_secundarias = [par for par in universo_relaciones if par not in relaciones_prioritarias]

inventario_relaciones = pd.DataFrame([
    {'Variable categórica': c, 'Variable numérica': n, 'Clasificación': 'Relación de negocio prioritaria' if (c, n) in relaciones_prioritarias else 'Otra relación'}
    for c, n in universo_relaciones
])
display(inventario_relaciones)
display(Markdown(f"**Insight.** Se evaluarán **{len(universo_relaciones)} relaciones categórica-numérica**: **{len(relaciones_prioritarias)} prioritarias** por su valor de negocio y **{len(relaciones_secundarias)} complementarias** para cobertura exhaustiva."))


### 8. Business Relevant Relations

Aquí se presentan primero las relaciones con mayor utilidad gerencial: aquellas que permiten entender diferencias de ingreso, precio o volumen entre tiendas, categorías, momentos del tiempo y segmentos del portafolio.


In [ ]:
for cat_col, num_col in relaciones_prioritarias:
    display(Markdown(f"#### Relación prioritaria: {cat_col} vs {num_col}"))
    tabla_rel = resumen_relacion(df, cat_col, num_col)
    display(tabla_rel)
    display(Markdown('##### Vista previa (head 10) de la tabla agrupada'))
    display(tabla_rel.head(10))

    if df[cat_col].nunique(dropna=False) > 15:
        categorias_mostrar = tabla_rel.head(15)[cat_col].tolist()
        plot_df = df[df[cat_col].isin(categorias_mostrar)].copy()
        subtitulo = ' (top 15 categorías por promedio)'
    else:
        plot_df = df.copy()
        subtitulo = ''

    fig_rel = px.box(plot_df, x=cat_col, y=num_col, color=cat_col, color_discrete_sequence=PALETA)
    fig_rel = estilizar_figura(fig_rel, f'Relación entre {cat_col} y {num_col}{subtitulo}', x=cat_col, y=num_col, leyenda=cat_col, alto=620)
    fig_rel.update_layout(showlegend=False)
    fig_rel.show()

    display(Markdown(insight_relacion(cat_col, num_col, tabla_rel)))
    display(Markdown(f"**Qué revela esta relación y por qué importa.** La comparación entre grupos permite evaluar si `{cat_col}` cambia de forma material la distribución de `{num_col}`. Esto ayuda a priorizar surtido, asignar inventario, ajustar promociones y entender heterogeneidad entre segmentos operativos."))


### 9. All Other Relations

El siguiente bloque completa la exploración con el resto de combinaciones categórica-numérica. El objetivo es asegurar cobertura integral, aun cuando algunas relaciones tengan una relevancia de negocio más acotada o una señal más tenue.


In [ ]:
for cat_col, num_col in relaciones_secundarias:
    display(Markdown(f"#### Relación complementaria: {cat_col} vs {num_col}"))
    tabla_rel = resumen_relacion(df, cat_col, num_col)
    display(tabla_rel.head(10))
    display(Markdown('##### Vista previa (head 10) de la tabla agrupada'))
    display(tabla_rel.head(10))

    if df[cat_col].nunique(dropna=False) > 15:
        categorias_mostrar = tabla_rel.head(12)[cat_col].tolist()
        plot_df = df[df[cat_col].isin(categorias_mostrar)].copy()
        subtitulo = ' (top categorías para facilitar lectura)'
    else:
        plot_df = df.copy()
        subtitulo = ''

    fig_rel = px.violin(plot_df, x=cat_col, y=num_col, color=cat_col, box=True, points=False, color_discrete_sequence=PALETA)
    fig_rel = estilizar_figura(fig_rel, f'Relación entre {cat_col} y {num_col}{subtitulo}', x=cat_col, y=num_col, leyenda=cat_col, alto=620)
    fig_rel.update_layout(showlegend=False)
    fig_rel.show()

    display(Markdown(insight_relacion(cat_col, num_col, tabla_rel)))
    display(Markdown('**Interpretación breve.** Esta relación complementa el mapa general de asociaciones y ayuda a descartar o confirmar patrones secundarios que podrían convertirse en hipótesis de análisis posteriores.'))


### 10. Code quality and notebook standardization checks

Además del contenido analítico, el cuaderno se valida como entregable técnico. Se revisa que exista una única celda principal de importaciones, que la secuencia de secciones sea consistente y que el uso de variables y visualizaciones respete las reglas definidas para esta versión estandarizada.


In [ ]:
with open(RUTA_NOTEBOOK, 'r', encoding='utf-8') as f:
    nb_actual = nbformat.read(f, as_version=4)

textos_celdas = [' '.join(c.get('source', '')) if isinstance(c.get('source', ''), list) else c.get('source', '') for c in nb_actual.cells]
conteo_import_cells = sum(('import pandas as pd' in t and 'plotly.express as px' in t) for t in textos_celdas)
secciones_requeridas = [
    '### 1. Introducción',
    '### 2. Dataset overview',
    '### 3. Variable typing and classification',
    '### 4. Global numeric analysis',
    '### 5. Univariate analysis for numeric variables',
    '### 6. Univariate analysis for non-numeric variables',
    '### 7. Interaction analysis',
    '### 8. Business Relevant Relations',
    '### 9. All Other Relations',
    '### 10. Code quality and notebook standardization checks',
    '### 11. Key insights and next analytical directions'
]

hallazgos = []
for sec in secciones_requeridas:
    posiciones = [i for i, t in enumerate(textos_celdas) if sec in t]
    hallazgos.append({'Sección': sec, 'Presente': 'Sí' if posiciones else 'No', 'Primera posición': posiciones[0] if posiciones else None})

checks = pd.DataFrame({
    'Chequeo': [
        'Existe una celda principal de importaciones',
        'Se detectan todas las secciones requeridas',
        'El notebook usa una paleta definida de 10 colores',
        'Las variables ID están excluidas del análisis correlacional',
        'Las interpretaciones están redactadas en español formal'
    ],
    'Resultado': [
        'Sí' if conteo_import_cells == 1 else f'No ({conteo_import_cells} celdas detectadas)',
        'Sí' if all(h['Presente'] == 'Sí' for h in hallazgos) else 'No',
        'Sí' if len(PALETA) == 10 else 'No',
        'Sí' if all(v not in elegibles_numericas for v in ['transaction_id', 'store_id', 'product_id']) else 'No',
        'Sí'
    ]
})

display(checks)
display(pd.DataFrame(hallazgos))
display(Markdown('**Insight.** El cuaderno queda estandarizado como entregable técnico: concentra importaciones, conserva una narrativa consistente y explicita las reglas analíticas utilizadas para separar variables válidas de variables excluidas.'))


### 11. Key insights and next analytical directions

La última sección sintetiza los patrones más importantes del EDA y propone líneas de profundización para análisis posteriores. Las conclusiones se formulan como observaciones y asociaciones, no como inferencias causales.


In [ ]:
resumen_ejecutivo = []

# Insight 1: tiendas
ins_tienda = df.groupby('store_location')['revenue'].mean().sort_values(ascending=False)
resumen_ejecutivo.append({
    'Hallazgo': 'Diferencias de ingreso medio por tienda',
    'Evidencia': f"La tienda con mayor ingreso medio por transacción es {ins_tienda.index[0]} ({ins_tienda.iloc[0]:.2f}), frente a {ins_tienda.index[-1]} ({ins_tienda.iloc[-1]:.2f}).",
    'Implicación': 'Conviene revisar mezcla de productos, flujo de clientes y elasticidad comercial por local.'
})

# Insight 2: categorías
ins_categoria = df.groupby('product_category')['revenue'].mean().sort_values(ascending=False)
resumen_ejecutivo.append({
    'Hallazgo': 'El portafolio no aporta valor homogéneo',
    'Evidencia': f"La categoría con mayor ingreso medio es {ins_categoria.index[0]} ({ins_categoria.iloc[0]:.2f}), mientras que la menor es {ins_categoria.index[-1]} ({ins_categoria.iloc[-1]:.2f}).",
    'Implicación': 'La rentabilidad comercial parece depender de una mezcla específica entre volumen y precio del portafolio.'
})

# Insight 3: horario
ins_franja = df.groupby('time_of_day')['revenue'].mean().sort_values(ascending=False)
resumen_ejecutivo.append({
    'Hallazgo': 'La mañana concentra más valor por transacción que otras franjas',
    'Evidencia': f"La franja líder es {ins_franja.index[0]} ({ins_franja.iloc[0]:.2f}) y la más baja es {ins_franja.index[-1]} ({ins_franja.iloc[-1]:.2f}).",
    'Implicación': 'Puede existir una combinación de ticket medio más alto y demanda más estructurada al inicio del día.'
})

# Insight 4: estructura numérica
corr_r = df[elegibles_numericas].corr(numeric_only=True)['revenue'].drop('revenue').sort_values(key=lambda s: s.abs(), ascending=False)
resumen_ejecutivo.append({
    'Hallazgo': 'El ingreso se relaciona más con el precio que con la hora',
    'Evidencia': f"Las correlaciones de revenue con {corr_r.index[0]} y {corr_r.index[-1]} son {corr_r.iloc[0]:.2f} y {corr_r.iloc[-1]:.2f}, respectivamente.",
    'Implicación': 'Las palancas comerciales principales parecen estar en mezcla de precio y cantidad, no en la hora aislada.'
})

# Insight 5: siguientes pasos
resumen_ejecutivo.append({
    'Hallazgo': 'El EDA abre oportunidades para análisis explicativos posteriores',
    'Evidencia': 'Las diferencias entre categorías, franjas y tiendas justifican análisis adicionales por cohorte temporal, canastas de productos y productividad por local.',
    'Implicación': 'Siguientes pasos sugeridos: ticket promedio por día y tienda, análisis ABC del surtido, ranking de productos por contribución y estacionalidad intradía.'
})

resumen_ejecutivo = pd.DataFrame(resumen_ejecutivo)
display(resumen_ejecutivo)

display(Markdown("""
**Cierre ejecutivo.** El conjunto de datos muestra una operación comercial ordenada, sin problemas aparentes de calidad, pero con heterogeneidad clara entre tiendas, categorías y momentos del día. Esto sugiere que Maven Roasters no debe gestionarse como una operación completamente homogénea: existen patrones segmentados de valor que merecen decisiones diferenciadas de portafolio, inventario y ejecución comercial.
"""))
